# MATH 5010 Computer Lab — Section 8  
## Sampling and Order Statistics  
### Full Solutions Included

This lab accompanies **Section 8: Sampling and Order Statistics**.

We will use Python to study:

1. Random samples and IID assumptions  
2. Sampling with and without replacement  
3. Statistics and sampling distributions  
4. Sample mean and sample variance  
5. Sums of random samples  
6. Sampling from a normal distribution  
7. Chi-square, Student's $t$, and $F$ distributions  
8. Order statistics  
9. Minimum, maximum, median, and sample range  
10. Practice problems with complete solutions

The main message is that a statistic is a random variable computed from a random sample, and its distribution is called a sampling distribution.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from math import sqrt

rng = np.random.default_rng(5010)

pd.set_option("display.precision", 5)
print("Packages loaded.")

## 1. Random Samples

A random sample $X_1,\ldots,X_n$ from a population distribution $f(x)$ means:

1. $X_1,\ldots,X_n$ are independent.
2. Each $X_i$ has the same distribution $f(x)$.

In this case the joint density or mass function is

\[
f(x_1,\ldots,x_n)=\prod_{i=1}^n f(x_i).
\]

We first simulate an IID sample from an exponential distribution.

In [ ]:
n = 10
beta = 2.0  # scale parameter
sample = rng.exponential(scale=beta, size=n)

pd.DataFrame({"i": np.arange(1, n+1), "X_i": sample})

In [ ]:
print("Sample mean:", sample.mean())
print("Sample variance with denominator n-1:", sample.var(ddof=1))
print("Sample variance with denominator n:", sample.var(ddof=0))

### Full Solution

If $X_i\sim \mathrm{Exponential}(\beta)$ independently, then

\[
f(x_i;\beta)=\frac{1}{\beta}e^{-x_i/\beta},\qquad x_i>0.
\]

The joint density is

\[
f(x_1,\ldots,x_n;\beta)
=
\prod_{i=1}^n \frac{1}{\beta}e^{-x_i/\beta}
=
\beta^{-n}\exp\left(-\frac{1}{\beta}\sum_{i=1}^n x_i\right),
\qquad x_i>0.
\]

## 2. Sampling With Replacement vs Without Replacement

Sampling with replacement from a finite population can produce IID observations.  
Sampling without replacement produces identically distributed observations, but not independent observations.

Let the population be

\[
\{1,2,\ldots,N\}.
\]

If we sample without replacement, knowing $X_1$ changes the distribution of $X_2$.

In [ ]:
population = np.arange(1, 11)

with_replacement = rng.choice(population, size=5, replace=True)
without_replacement = rng.choice(population, size=5, replace=False)

print("Sample with replacement:", with_replacement)
print("Sample without replacement:", without_replacement)

In [ ]:
# Demonstrate dependence without replacement.
N = 10
x = 3
y = 7

p_X2_y_given_X1_x = 1/(N-1)
p_X2_y_given_X1_y = 0
p_X2_y_marginal = 1/N

pd.DataFrame({
    "Quantity": [
        "P(X2=y)",
        "P(X2=y | X1=x), x != y",
        "P(X2=y | X1=y)"
    ],
    "Value": [
        p_X2_y_marginal,
        p_X2_y_given_X1_x,
        p_X2_y_given_X1_y
    ]
})

### Full Solution

Without replacement,

\[
P(X_2=y)=\frac1N.
\]

If $X_1=x$ and $x\ne y$, then $y$ is still available among $N-1$ values, so

\[
P(X_2=y\mid X_1=x)=\frac1{N-1}.
\]

But if $X_1=y$, then $y$ is no longer available, so

\[
P(X_2=y\mid X_1=y)=0.
\]

Since these conditional probabilities differ from $P(X_2=y)$, $X_1$ and $X_2$ are not independent.

## 3. Statistics and Sampling Distributions

A statistic is any function of the sample:

\[
T=T(X_1,\ldots,X_n).
\]

Examples:

\[
\bar X=\frac1n\sum_{i=1}^n X_i,
\qquad
S^2=\frac{1}{n-1}\sum_{i=1}^n (X_i-\bar X)^2,
\qquad
X_{(n)}=\max(X_1,\ldots,X_n).
\]

The distribution of a statistic is called its sampling distribution.

In [ ]:
# Sampling distribution of the sample mean from an exponential population
reps = 50_000
n = 30
beta = 2.0

samples = rng.exponential(scale=beta, size=(reps, n))
sample_means = samples.mean(axis=1)
sample_variances = samples.var(axis=1, ddof=1)
sample_maxima = samples.max(axis=1)

summary = pd.DataFrame({
    "Statistic": ["Xbar", "S^2", "Max"],
    "Simulated mean": [sample_means.mean(), sample_variances.mean(), sample_maxima.mean()],
    "Simulated variance": [sample_means.var(ddof=0), sample_variances.var(ddof=0), sample_maxima.var(ddof=0)]
})
display(summary)

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(sample_means, bins=60, density=True, alpha=0.7)
plt.xlabel(r"$\bar X$")
plt.ylabel("Density")
plt.title("Sampling Distribution of Sample Mean, Exponential Population")
plt.show()

### Full Solution

For an exponential population with scale $\beta$,

\[
E[X]=\beta,\qquad \operatorname{Var}(X)=\beta^2.
\]

Therefore,

\[
E[\bar X]=\beta,\qquad \operatorname{Var}(\bar X)=\frac{\beta^2}{n}.
\]

The simulation estimates the sampling distribution of $\bar X$ by repeatedly drawing samples and computing the statistic.

## 4. Sample Mean and Sample Variance

For a random sample from a population with mean $\mu$ and variance $\sigma^2$,

\[
E[\bar X]=\mu,
\qquad
\operatorname{Var}(\bar X)=\frac{\sigma^2}{n},
\qquad
E[S^2]=\sigma^2.
\]

Here

\[
S^2=\frac{1}{n-1}\sum_{i=1}^n (X_i-\bar X)^2.
\]

We verify these formulas by simulation.

In [ ]:
reps = 80_000
n_values = [5, 10, 30, 100]
mu = 3
sigma = 2

rows = []
for n in n_values:
    samples = rng.normal(mu, sigma, size=(reps, n))
    xbar = samples.mean(axis=1)
    s2_unbiased = samples.var(axis=1, ddof=1)
    s2_biased = samples.var(axis=1, ddof=0)
    rows.append([
        n,
        xbar.mean(),
        xbar.var(ddof=0),
        sigma**2/n,
        s2_unbiased.mean(),
        s2_biased.mean()
    ])

pd.DataFrame(rows, columns=[
    "n", "Mean of Xbar", "Var of Xbar", "Theory sigma^2/n",
    "Mean of S^2 (ddof=1)", "Mean of biased variance (ddof=0)"
])

### Full Solution

Because $\bar X=(X_1+\cdots+X_n)/n$,

\[
E[\bar X]=\frac1n\sum_{i=1}^n E[X_i]=\mu.
\]

Since the $X_i$ are independent,

\[
\operatorname{Var}(\bar X)
=
\frac1{n^2}\sum_{i=1}^n \operatorname{Var}(X_i)
=
\frac{\sigma^2}{n}.
\]

The statistic

\[
S^2=\frac{1}{n-1}\sum_{i=1}^n (X_i-\bar X)^2
\]

is unbiased:

\[
E[S^2]=\sigma^2.
\]

The version with denominator $n$ is biased downward.

## 5. Sum of Random Samples and the MGF Relation

Let

\[
Y=X_1+\cdots+X_n.
\]

If $X_1,\ldots,X_n$ are independent and each has MGF $M_X(t)$, then

\[
M_Y(t)=\left(M_X(t)\right)^n.
\]

For the sample mean,

\[
\bar X=\frac{Y}{n},
\]

so

\[
M_{\bar X}(t)=\left(M_X(t/n)\right)^n.
\]

### Example: Normal Sample Mean

If $X_i\sim N(\mu,\sigma^2)$, then

\[
\bar X\sim N\left(\mu,\frac{\sigma^2}{n}\right).
\]

In [ ]:
reps = 80_000
n = 25
mu = 5
sigma = 3

samples = rng.normal(mu, sigma, size=(reps, n))
xbar = samples.mean(axis=1)

grid = np.linspace(mu - 4*sigma/sqrt(n), mu + 4*sigma/sqrt(n), 300)
theory_pdf = stats.norm.pdf(grid, loc=mu, scale=sigma/sqrt(n))

plt.figure(figsize=(7, 4))
plt.hist(xbar, bins=60, density=True, alpha=0.7, label="Simulation")
plt.plot(grid, theory_pdf, label=r"$N(\mu,\sigma^2/n)$ theory")
plt.xlabel(r"$\bar X$")
plt.ylabel("Density")
plt.title("Normal Sample Mean")
plt.legend()
plt.show()

### Full Solution

The MGF of $X_i\sim N(\mu,\sigma^2)$ is

\[
M_X(t)=\exp\left(\mu t+\frac{\sigma^2t^2}{2}\right).
\]

Thus

\[
M_{\bar X}(t)
=
\left(M_X(t/n)\right)^n
=
\left[
\exp\left(\mu\frac{t}{n}+\frac{\sigma^2(t/n)^2}{2}\right)
\right]^n
=
\exp\left(\mu t+\frac{\sigma^2 t^2}{2n}\right).
\]

This is the MGF of

\[
N\left(\mu,\frac{\sigma^2}{n}\right).
\]

## 6. Gamma Sums and Sample Means

If

\[
X_i\sim \mathrm{Gamma}(\alpha,\beta)
\]

with scale parameter $\beta$, then

\[
Y=\sum_{i=1}^n X_i\sim \mathrm{Gamma}(n\alpha,\beta).
\]

Therefore,

\[
\bar X=\frac{Y}{n}\sim \mathrm{Gamma}(n\alpha,\beta/n).
\]

In [ ]:
reps = 80_000
n = 10
alpha = 2.0
beta = 3.0  # scale

samples = rng.gamma(shape=alpha, scale=beta, size=(reps, n))
sums = samples.sum(axis=1)
means = samples.mean(axis=1)

grid_sum = np.linspace(0, np.quantile(sums, 0.995), 400)
pdf_sum = stats.gamma.pdf(grid_sum, a=n*alpha, scale=beta)

plt.figure(figsize=(7, 4))
plt.hist(sums, bins=60, density=True, alpha=0.7, label="Simulated sums")
plt.plot(grid_sum, pdf_sum, label=r"$\Gamma(n\alpha,\beta)$ theory")
plt.xlabel(r"$Y=\sum X_i$")
plt.ylabel("Density")
plt.title("Sum of Gamma Random Variables")
plt.legend()
plt.show()

print("Simulated mean of Xbar:", means.mean())
print("Theoretical E[Xbar]:", alpha*beta)
print("Simulated Var(Xbar):", means.var(ddof=0))
print("Theoretical Var(Xbar):", alpha*beta**2/n)

### Full Solution

The MGF of $\mathrm{Gamma}(\alpha,\beta)$ with scale $\beta$ is

\[
M_X(t)=(1-\beta t)^{-\alpha}.
\]

For the sum,

\[
M_Y(t)=\left(M_X(t)\right)^n
=
(1-\beta t)^{-n\alpha},
\]

so

\[
Y\sim \mathrm{Gamma}(n\alpha,\beta).
\]

For $\bar X=Y/n$, the scale is divided by $n$:

\[
\bar X\sim \mathrm{Gamma}(n\alpha,\beta/n).
\]

## 7. Convolution: Sum of Two Uniform Random Variables

Let

\[
X,Y\sim \mathrm{Uniform}(0,1)
\]

independently and let

\[
Z=X+Y.
\]

The density of $Z$ is the convolution

\[
f_Z(z)=\int_{-\infty}^{\infty} f_X(w)f_Y(z-w)\,dw.
\]

For two independent uniform random variables,

\[
f_Z(z)=
\begin{cases}
z, & 0<z<1,\\
2-z, & 1\le z<2,\\
0, & \text{otherwise}.
\end{cases}
\]

In [ ]:
Nsim = 200_000
X = rng.uniform(0, 1, size=Nsim)
Y = rng.uniform(0, 1, size=Nsim)
Z = X + Y

grid = np.linspace(0, 2, 400)
pdf = np.where(grid < 1, grid, 2-grid)
pdf = np.maximum(pdf, 0)

plt.figure(figsize=(7, 4))
plt.hist(Z, bins=80, density=True, alpha=0.7, label="Simulation")
plt.plot(grid, pdf, label="Convolution theory")
plt.xlabel("z")
plt.ylabel("Density")
plt.title("Convolution: Sum of Two Uniform(0,1) Variables")
plt.legend()
plt.show()

### Full Solution

For $0<z<1$, we need $0<w<1$ and $0<z-w<1$. This gives $0<w<z$, so

\[
f_Z(z)=\int_0^z 1\,dw=z.
\]

For $1\le z<2$, the constraints give $z-1<w<1$, so

\[
f_Z(z)=\int_{z-1}^1 1\,dw=2-z.
\]

## 8. Sampling from a Normal Distribution

If

\[
X_1,\ldots,X_n\sim N(\mu,\sigma^2),
\]

then

\[
\bar X\sim N\left(\mu,\frac{\sigma^2}{n}\right),
\]

and

\[
\frac{(n-1)S^2}{\sigma^2}\sim \chi^2_{n-1}.
\]

Also, for normal samples,

\[
\bar X \text{ and } S^2
\]

are independent.

In [ ]:
reps = 100_000
n = 12
mu = 10
sigma = 4

samples = rng.normal(mu, sigma, size=(reps, n))
xbar = samples.mean(axis=1)
s2 = samples.var(axis=1, ddof=1)
chi_stat = (n-1)*s2/(sigma**2)

print("Correlation between Xbar and S^2:", np.corrcoef(xbar, s2)[0,1])
print("Mean of chi-square statistic:", chi_stat.mean())
print("Theoretical chi-square mean:", n-1)
print("Variance of chi-square statistic:", chi_stat.var(ddof=0))
print("Theoretical chi-square variance:", 2*(n-1))

In [ ]:
grid = np.linspace(0, stats.chi2.ppf(0.995, df=n-1), 400)

plt.figure(figsize=(7, 4))
plt.hist(chi_stat, bins=70, density=True, alpha=0.7, label="Simulation")
plt.plot(grid, stats.chi2.pdf(grid, df=n-1), label=fr"$\chi^2_{{{n-1}}}$ theory")
plt.xlabel(r"$(n-1)S^2/\sigma^2$")
plt.ylabel("Density")
plt.title("Sampling Distribution of Sample Variance")
plt.legend()
plt.show()

### Full Solution

For normal samples,

\[
\bar X\sim N\left(\mu,\frac{\sigma^2}{n}\right),
\]

and

\[
\frac{(n-1)S^2}{\sigma^2}\sim \chi^2_{n-1}.
\]

Moreover, $\bar X$ and $S^2$ are independent.  
The simulation checks this by computing the empirical correlation between $\bar X$ and $S^2$, which should be close to zero.

## 9. Student's $t$ Distribution

If

\[
Z\sim N(0,1),
\qquad
V\sim \chi^2_\nu,
\]

and $Z$ and $V$ are independent, then

\[
T=\frac{Z}{\sqrt{V/\nu}}\sim t_\nu.
\]

For a normal sample,

\[
T=
\frac{\bar X-\mu}{S/\sqrt n}
\sim t_{n-1}.
\]

In [ ]:
reps = 100_000
n = 8
mu = 5
sigma = 2

samples = rng.normal(mu, sigma, size=(reps, n))
xbar = samples.mean(axis=1)
s = samples.std(axis=1, ddof=1)

t_stat = (xbar - mu) / (s / np.sqrt(n))

grid = np.linspace(stats.t.ppf(0.001, df=n-1), stats.t.ppf(0.999, df=n-1), 400)

plt.figure(figsize=(7, 4))
plt.hist(t_stat, bins=80, density=True, alpha=0.7, label="Simulation")
plt.plot(grid, stats.t.pdf(grid, df=n-1), label=fr"$t_{{{n-1}}}$ theory")
plt.xlabel("t statistic")
plt.ylabel("Density")
plt.title("Student's t Distribution from a Normal Sample")
plt.legend()
plt.show()

### Full Solution

For a normal sample,

\[
\frac{\bar X-\mu}{\sigma/\sqrt n}\sim N(0,1),
\]

and

\[
\frac{(n-1)S^2}{\sigma^2}\sim \chi^2_{n-1}.
\]

These two random variables are independent. Therefore,

\[
\frac{\bar X-\mu}{S/\sqrt n}
=
\frac{(\bar X-\mu)/(\sigma/\sqrt n)}
{\sqrt{[(n-1)S^2/\sigma^2]/(n-1)}}
\sim t_{n-1}.
\]

## 10. The $F$ Distribution

If

\[
U\sim \chi^2_p,\qquad V\sim \chi^2_q
\]

independently, then

\[
F=\frac{U/p}{V/q}\sim F_{p,q}.
\]

The $F$ distribution is important for comparing variances and for ANOVA.

In [ ]:
reps = 100_000
p_df = 5
q_df = 12

U = rng.chisquare(df=p_df, size=reps)
V = rng.chisquare(df=q_df, size=reps)

F_stat = (U/p_df) / (V/q_df)

grid = np.linspace(0, stats.f.ppf(0.995, p_df, q_df), 400)

plt.figure(figsize=(7, 4))
plt.hist(F_stat, bins=80, density=True, alpha=0.7, label="Simulation")
plt.plot(grid, stats.f.pdf(grid, p_df, q_df), label=fr"$F_{{{p_df},{q_df}}}$ theory")
plt.xlabel("F")
plt.ylabel("Density")
plt.title("F Distribution as Ratio of Chi-square Variables")
plt.legend()
plt.show()

In [ ]:
# Verify common identities:
# If T ~ t_q, then T^2 ~ F_{1,q}
q = 10
T = rng.standard_t(df=q, size=reps)
T2 = T**2

grid = np.linspace(0, stats.f.ppf(0.995, 1, q), 400)

plt.figure(figsize=(7, 4))
plt.hist(T2, bins=80, density=True, alpha=0.7, label=r"Simulation of $T^2$")
plt.plot(grid, stats.f.pdf(grid, 1, q), label=fr"$F_{{1,{q}}}$ theory")
plt.xlabel(r"$T^2$")
plt.ylabel("Density")
plt.title(r"If $T\sim t_q$, then $T^2\sim F_{1,q}$")
plt.legend()
plt.show()

### Full Solution

By definition,

\[
F=\frac{U/p}{V/q}
\]

has an $F_{p,q}$ distribution when $U$ and $V$ are independent chi-square variables with $p$ and $q$ degrees of freedom.

Also, if

\[
T=\frac{Z}{\sqrt{V/q}}\sim t_q,
\]

then

\[
T^2=\frac{Z^2/1}{V/q}.
\]

Since $Z^2\sim \chi^2_1$, this gives

\[
T^2\sim F_{1,q}.
\]

## 11. Order Statistics

Let

\[
X_1,\ldots,X_n
\]

be a random sample. The order statistics are

\[
X_{(1)}\le X_{(2)}\le \cdots \le X_{(n)}.
\]

Here:

\[
X_{(1)}=\min(X_1,\ldots,X_n),
\qquad
X_{(n)}=\max(X_1,\ldots,X_n).
\]

For a continuous population with CDF $F$ and PDF $f$, the PDF of the $j$-th order statistic is

\[
f_{X_{(j)}}(x)
=
\frac{n!}{(j-1)!(n-j)!}
[F(x)]^{j-1}
[1-F(x)]^{n-j}
f(x).
\]

In [ ]:
# Simulate order statistics for Uniform(0,1)
reps = 100_000
n = 10
j = 3

samples = rng.uniform(0, 1, size=(reps, n))
ordered = np.sort(samples, axis=1)
Xj = ordered[:, j-1]  # j-th order statistic

grid = np.linspace(0, 1, 400)
theory_pdf = stats.beta.pdf(grid, a=j, b=n+1-j)

plt.figure(figsize=(7, 4))
plt.hist(Xj, bins=70, density=True, alpha=0.7, label=f"Simulation of X_({j})")
plt.plot(grid, theory_pdf, label=fr"Beta({j}, {n+1-j}) theory")
plt.xlabel("x")
plt.ylabel("Density")
plt.title("Order Statistic from Uniform(0,1)")
plt.legend()
plt.show()

### Full Solution

For $X_i\sim \mathrm{Uniform}(0,1)$,

\[
F(x)=x,\qquad f(x)=1,\qquad 0<x<1.
\]

Therefore,

\[
f_{X_{(j)}}(x)
=
\frac{n!}{(j-1)!(n-j)!}
x^{j-1}(1-x)^{n-j},
\qquad 0<x<1.
\]

This is the density of a Beta distribution:

\[
X_{(j)}\sim \mathrm{Beta}(j,n+1-j).
\]

## 12. Mean and Variance of Uniform Order Statistics

If

\[
X_{(j)}\sim \mathrm{Beta}(j,n+1-j),
\]

then

\[
E[X_{(j)}]=\frac{j}{n+1},
\]

and

\[
\operatorname{Var}(X_{(j)})
=
\frac{j(n+1-j)}{(n+1)^2(n+2)}.
\]

In [ ]:
n = 10
reps = 200_000
samples = rng.uniform(0, 1, size=(reps, n))
ordered = np.sort(samples, axis=1)

rows = []
for j in [1, 2, 5, 10]:
    Xj = ordered[:, j-1]
    theory_mean = j/(n+1)
    theory_var = j*(n+1-j)/((n+1)**2*(n+2))
    rows.append([j, Xj.mean(), theory_mean, Xj.var(ddof=0), theory_var])

pd.DataFrame(rows, columns=[
    "j", "Simulated E[X_(j)]", "Theory E[X_(j)]",
    "Simulated Var[X_(j)]", "Theory Var[X_(j)]"
])

### Full Solution

For a Beta$(\alpha,\beta)$ random variable,

\[
E[X]=\frac{\alpha}{\alpha+\beta},
\]

and

\[
\operatorname{Var}(X)
=
\frac{\alpha\beta}{(\alpha+\beta)^2(\alpha+\beta+1)}.
\]

Here $\alpha=j$ and $\beta=n+1-j$, so

\[
E[X_{(j)}]=\frac{j}{n+1},
\]

and

\[
\operatorname{Var}(X_{(j)})
=
\frac{j(n+1-j)}{(n+1)^2(n+2)}.
\]

## 13. Minimum and Maximum

For continuous IID samples:

\[
P(X_{(1)}>x)=P(X_1>x,\ldots,X_n>x)=[1-F(x)]^n,
\]

so

\[
F_{X_{(1)}}(x)=1-[1-F(x)]^n.
\]

For the maximum,

\[
F_{X_{(n)}}(x)=P(X_1\le x,\ldots,X_n\le x)=[F(x)]^n.
\]

For $X_i\sim U(0,1)$,

\[
X_{(1)}\sim \mathrm{Beta}(1,n),
\qquad
X_{(n)}\sim \mathrm{Beta}(n,1).
\]

In [ ]:
n = 20
reps = 150_000

samples = rng.uniform(0, 1, size=(reps, n))
mins = samples.min(axis=1)
maxs = samples.max(axis=1)

grid = np.linspace(0, 1, 400)

plt.figure(figsize=(7, 4))
plt.hist(mins, bins=70, density=True, alpha=0.6, label="Simulation of min")
plt.plot(grid, stats.beta.pdf(grid, 1, n), label=r"Beta(1,n) theory")
plt.xlabel("x")
plt.ylabel("Density")
plt.title("Minimum of Uniform Sample")
plt.legend()
plt.show()

plt.figure(figsize=(7, 4))
plt.hist(maxs, bins=70, density=True, alpha=0.6, label="Simulation of max")
plt.plot(grid, stats.beta.pdf(grid, n, 1), label=r"Beta(n,1) theory")
plt.xlabel("x")
plt.ylabel("Density")
plt.title("Maximum of Uniform Sample")
plt.legend()
plt.show()

### Full Solution

For $U(0,1)$, $F(x)=x$ for $0<x<1$.

Thus

\[
F_{X_{(1)}}(x)=1-(1-x)^n,
\]

and differentiating gives

\[
f_{X_{(1)}}(x)=n(1-x)^{n-1},
\]

which is Beta$(1,n)$.

For the maximum,

\[
F_{X_{(n)}}(x)=x^n,
\]

so

\[
f_{X_{(n)}}(x)=nx^{n-1},
\]

which is Beta$(n,1)$.

## 14. Sample Range

The sample range is

\[
R=X_{(n)}-X_{(1)}.
\]

For a Uniform$(0,1)$ sample, the range has density

\[
f_R(r)=n(n-1)r^{n-2}(1-r),
\qquad 0<r<1.
\]

This is a useful example of a statistic built from two order statistics.

In [ ]:
n = 8
reps = 150_000

samples = rng.uniform(0, 1, size=(reps, n))
ranges = samples.max(axis=1) - samples.min(axis=1)

grid = np.linspace(0, 1, 400)
range_pdf = n*(n-1)*(grid**(n-2))*(1-grid)

plt.figure(figsize=(7, 4))
plt.hist(ranges, bins=70, density=True, alpha=0.7, label="Simulation")
plt.plot(grid, range_pdf, label="Theory")
plt.xlabel("r")
plt.ylabel("Density")
plt.title("Sample Range for Uniform(0,1)")
plt.legend()
plt.show()

print("Simulated E[Range]:", ranges.mean())
print("Theoretical E[Range]:", (n-1)/(n+1))

### Full Solution

The joint density of the minimum $U=X_{(1)}$ and maximum $V=X_{(n)}$ for a Uniform$(0,1)$ sample is

\[
f_{U,V}(u,v)=n(n-1)(v-u)^{n-2},
\qquad 0<u<v<1.
\]

Let $R=V-U$. For a fixed $r$, $u$ ranges from $0$ to $1-r$. Hence

\[
f_R(r)=\int_0^{1-r} n(n-1)r^{n-2}\,du
=
n(n-1)r^{n-2}(1-r),
\qquad 0<r<1.
\]

The mean is

\[
E[R]=\frac{n-1}{n+1}.
\]

# Practice Problems with Full Solutions

## Practice Problem 1 — Sample Mean and Variance

Let $X_1,\ldots,X_n$ be IID with

\[
E[X_i]=10,\qquad \operatorname{Var}(X_i)=9.
\]

For $n=36$, find

\[
E[\bar X],\qquad \operatorname{Var}(\bar X).
\]

In [ ]:
mu = 10
sigma2 = 9
n = 36

print("E[Xbar] =", mu)
print("Var(Xbar) =", sigma2/n)
print("SD(Xbar) =", np.sqrt(sigma2/n))

### Solution

For any IID sample,

\[
E[\bar X]=\mu,
\qquad
\operatorname{Var}(\bar X)=\frac{\sigma^2}{n}.
\]

Here $\mu=10$, $\sigma^2=9$, and $n=36$, so

\[
E[\bar X]=10,
\]

and

\[
\operatorname{Var}(\bar X)=\frac{9}{36}=\frac14.
\]

## Practice Problem 2 — Normal Sampling Theory

Suppose

\[
X_1,\ldots,X_{16}\sim N(100,25).
\]

Find the distribution of $\bar X$.

In [ ]:
mu = 100
sigma2 = 25
sigma = np.sqrt(sigma2)
n = 16

print("Mean of Xbar:", mu)
print("Variance of Xbar:", sigma2/n)
print("Standard deviation of Xbar:", sigma/np.sqrt(n))

### Solution

If $X_i\sim N(\mu,\sigma^2)$ independently, then

\[
\bar X\sim N\left(\mu,\frac{\sigma^2}{n}\right).
\]

Here $\mu=100$, $\sigma^2=25$, and $n=16$, so

\[
\bar X\sim N\left(100,\frac{25}{16}\right).
\]

The standard deviation of $\bar X$ is

\[
\sqrt{\frac{25}{16}}=\frac54=1.25.
\]

## Practice Problem 3 — Chi-square Statistic

Suppose

\[
X_1,\ldots,X_{10}\sim N(\mu,\sigma^2).
\]

Find the distribution of

\[
\frac{9S^2}{\sigma^2}.
\]

In [ ]:
df = 9
print("Distribution: chi-square with df =", df)
print("Mean:", df)
print("Variance:", 2*df)

### Solution

For a normal sample,

\[
\frac{(n-1)S^2}{\sigma^2}\sim \chi^2_{n-1}.
\]

Here $n=10$, so

\[
\frac{9S^2}{\sigma^2}\sim \chi^2_9.
\]

## Practice Problem 4 — Student's t Statistic

Suppose

\[
X_1,\ldots,X_{20}\sim N(\mu,\sigma^2).
\]

Find the distribution of

\[
T=\frac{\bar X-\mu}{S/\sqrt{20}}.
\]

In [ ]:
n = 20
print("Distribution: Student t with df =", n-1)

### Solution

For a normal sample with unknown variance,

\[
T=\frac{\bar X-\mu}{S/\sqrt n}\sim t_{n-1}.
\]

Here $n=20$, so

\[
T\sim t_{19}.
\]

## Practice Problem 5 — F Distribution

Let

\[
U\sim\chi^2_6,\qquad V\sim\chi^2_{10},
\]

independently. Find the distribution of

\[
\frac{U/6}{V/10}.
\]

In [ ]:
print("Distribution: F with numerator df 6 and denominator df 10")
print("Example 95th percentile:", stats.f.ppf(0.95, 6, 10))

### Solution

By definition,

\[
F=\frac{U/p}{V/q}\sim F_{p,q}
\]

when $U\sim\chi^2_p$, $V\sim\chi^2_q$, and $U,V$ are independent.

Here $p=6$ and $q=10$, so

\[
\frac{U/6}{V/10}\sim F_{6,10}.
\]

## Practice Problem 6 — Uniform Order Statistic

Let

\[
X_1,\ldots,X_5\sim U(0,1).
\]

Find the distribution, mean, and variance of the median $X_{(3)}$.

In [ ]:
n = 5
j = 3

alpha = j
beta_param = n + 1 - j

mean = alpha / (alpha + beta_param)
var = alpha * beta_param / ((alpha + beta_param)**2 * (alpha + beta_param + 1))

print(f"X_(3) ~ Beta({alpha}, {beta_param})")
print("Mean:", mean)
print("Variance:", var)

### Solution

For a Uniform$(0,1)$ sample,

\[
X_{(j)}\sim \mathrm{Beta}(j,n+1-j).
\]

Here $n=5$ and $j=3$, so

\[
X_{(3)}\sim \mathrm{Beta}(3,3).
\]

The mean is

\[
E[X_{(3)}]=\frac{3}{6}=\frac12.
\]

The variance is

\[
\operatorname{Var}(X_{(3)})
=
\frac{3(6-3)}{6^2(7)}
=
\frac{9}{252}
=
\frac1{28}.
\]

## Practice Problem 7 — Minimum and Maximum

Let

\[
X_1,\ldots,X_n\sim U(0,1).
\]

Find

\[
E[X_{(1)}],\qquad E[X_{(n)}].
\]

Evaluate these for $n=10$.

In [ ]:
n = 10
print("E[min] =", 1/(n+1))
print("E[max] =", n/(n+1))

### Solution

For Uniform$(0,1)$ order statistics,

\[
X_{(1)}\sim \mathrm{Beta}(1,n),
\qquad
X_{(n)}\sim \mathrm{Beta}(n,1).
\]

Therefore,

\[
E[X_{(1)}]=\frac{1}{n+1},
\qquad
E[X_{(n)}]=\frac{n}{n+1}.
\]

For $n=10$,

\[
E[X_{(1)}]=\frac1{11}\approx 0.0909,
\qquad
E[X_{(n)}]=\frac{10}{11}\approx 0.9091.
\]

## Practice Problem 8 — German Tank Style Estimator

Suppose tank serial numbers are uniformly distributed on

\[
\{1,2,\ldots,N\}.
\]

A sample of size $n$ is observed without replacement, and the maximum observed serial number is $M$.

A classical estimator is

\[
\hat N=\frac{n+1}{n}M-1.
\]

Simulate this estimator when $N=1000$ and $n=20$.

In [ ]:
N_true = 1000
n = 20
reps = 50_000

estimates = []
for _ in range(reps):
    sample = rng.choice(np.arange(1, N_true+1), size=n, replace=False)
    M = sample.max()
    N_hat = ((n+1)/n)*M - 1
    estimates.append(N_hat)

estimates = np.array(estimates)

print("True N:", N_true)
print("Mean of estimator:", estimates.mean())
print("Standard deviation of estimator:", estimates.std(ddof=0))

plt.figure(figsize=(7, 4))
plt.hist(estimates, bins=60, density=True, alpha=0.7)
plt.axvline(N_true, linestyle="--", label="True N")
plt.xlabel(r"$\hat N$")
plt.ylabel("Density")
plt.title("German Tank Estimator Simulation")
plt.legend()
plt.show()

### Solution

For sampling without replacement from $\{1,\ldots,N\}$, the expected maximum is

\[
E[M]=\frac{n(N+1)}{n+1}.
\]

Solving this expression for $N$ suggests

\[
N+1=\frac{n+1}{n}E[M].
\]

Replacing $E[M]$ by the observed maximum $M$ gives

\[
\hat N=\frac{n+1}{n}M-1.
\]

The simulation shows that this estimator is approximately unbiased.

# Summary

In this lab, we studied the computational side of Section 8:

- Random samples and IID structure
- Sampling with and without replacement
- Statistics and sampling distributions
- Sample mean and sample variance
- Sums and MGFs
- Convolution
- Normal sampling theory
- Chi-square, Student's $t$, and $F$ distributions
- Order statistics
- Minimum, maximum, range, and Uniform order-statistic formulas

The central message is:

\[
\boxed{\text{Statistics are random variables, and their sampling distributions drive inference.}}
\]